# Agent Tool Routing & Grounding Validation

**Goal**: Validate tool routing decisions and grounding behavior (entity provenance, hallucination checks).

This notebook provides:
- Single query deep dive (full message chain inspection)
- Basic hallucination detection (verify only retrieved result entities are in final answer)
- Batch analysis (7 test queries across different types)
- Aggregate metrics (tool usage, selection rates, error rates)

**NOTE:** This notebook is designed for a breast cancer-focused knowledge graph (query: "breast cancer AND targeted therapy", limit 20K abstracts)

In [1]:
# Setup

import os
import re
import pandas as pd
from typing import Any, Optional

import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv

from neo4j import GraphDatabase
from langchain_core.messages import HumanMessage

from biomed_kg_agent.agent.core import BiomedKGAgent

# Load environment
load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

if not NEO4J_PASSWORD:
    raise RuntimeError("NEO4J_PASSWORD environment variable is required for this notebook")

In [2]:
# Connect to Neo4j and initialize the agent (verbose=True)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
# Sanity check connection
with driver.session() as session:
    _ = session.run("RETURN 1").single()

agent = BiomedKGAgent(
    neo4j_driver=driver,
    model="claude-haiku-4-5-20251001",
    neo4j_database=os.getenv("NEO4J_DATABASE", "neo4j"),
    verbose=True,  # needed for tool and selection info
    temperature=0.1,
    min_evidence=5,
    max_results=20,
    enable_umls_linking=True,
)

print("Agent ready")


Agent ready


In [3]:
# Single query deep dive - full message chain inspection
question = "What drugs are used to treat breast cancer?"
# question = "Explain the relationship between HER2 and breast cancer"
response = agent.agent.invoke({"messages": [HumanMessage(content=question)]})

print("COMPLETE MESSAGE CHAIN")
print("=" * 80)

for i, msg in enumerate(response["messages"]):
    msg_type = msg.__class__.__name__
    print(f"\n[Message {i}] {msg_type}")
    print("-" * 40)
    
    if hasattr(msg, 'content') and msg.content:
        content = msg.content
        if isinstance(content, list):
            print(f"Content (structured, {len(content)} blocks):")
            print(content)
        # Truncate very long tool outputs for readability
        elif len(content) > 2000 and msg_type == "ToolMessage":
            print(f"Content ({len(content)} chars, truncated):")
            print(content[:1500] + "\n... [truncated] ...")
        else:
            print(f"Content ({len(content)} chars):")
            print(content)
    
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print("\nTool Calls:")
        for tc in msg.tool_calls:
            print(f"  {tc.get('name')}: {tc.get('args', {})}")

# Summary
print("\n" + "=" * 80)
print("SUMMARY")
print("-" * 40)

# Extract tool messages
tool_messages = [m for m in response["messages"] if m.__class__.__name__ == "ToolMessage"]

# Get final answer
final_answer = response["messages"][-1].content if response["messages"] else ""

# Count bullets only in DETAILED RESULTS section
detailed_section = final_answer.split("## DETAILED RESULTS")[-1] if "## DETAILED RESULTS" in final_answer else ""
bullets = detailed_section.count("\n- **")

# Count unique entity names from all tool outputs
unique_entities = set()
for tm in tool_messages:
    if hasattr(tm, 'content'):
        for match in re.finditer(r'ENTITY NAME \(use this exact name\): (.+)$', tm.content, re.MULTILINE):
            unique_entities.add(match.group(1).strip().lower())

print(f"Entities selected: {bullets}")
print(f"Tool calls made: {len(tool_messages)}")
print(f"Unique KG entities: {len(unique_entities)}")
if len(unique_entities) > 0:
    print(f"Selection rate: {bullets}/{len(unique_entities)} ({bullets/len(unique_entities)*100:.1f}%)")

Tool: FindEntityNeighbors | Input: breast cancer, Type: chemical
COMPLETE MESSAGE CHAIN

[Message 0] HumanMessage
----------------------------------------
Content (43 chars):
What drugs are used to treat breast cancer?

[Message 1] AIMessage
----------------------------------------
Content (structured, 2 blocks):
[{'text': "I'll search for drugs associated with breast cancer treatment.", 'type': 'text'}, {'id': 'toolu_01UvaaoZaLrgoRJQui35q6Tn', 'input': {'entity': 'breast cancer', 'entity_type': 'chemical'}, 'name': 'FindEntityNeighbors', 'type': 'tool_use', 'caller': {'type': 'direct'}}]

Tool Calls:
  FindEntityNeighbors: {'entity': 'breast cancer', 'entity_type': 'chemical'}

[Message 2] ToolMessage
----------------------------------------
Content (30895 chars, truncated):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESULT 1 (ISOLATED - DO NOT MIX WITH OTHER RESULTS)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ENTITY NAME (use this exact name): estrogen

Documents

In [4]:
# Test different query types to observe tool selection patterns
examples = [
    "What genes are associated with breast cancer?",
    "List chemicals related to breast cancer", 
    "Explain the relationship between HER2 and breast cancer",
]

for q in examples:
    print("\n" + "="*80)
    print(f"Query: {q}\n")
    resp = agent.agent.invoke({"messages": [HumanMessage(content=q)]})
    
    final = resp["messages"][-1].content if resp["messages"] else ""
    
    # Count bullets in DETAILED RESULTS section only
    detailed = final.split("## DETAILED RESULTS")[-1] if "## DETAILED RESULTS" in final else ""
    bullets = detailed.count("\n- **")
    
    print(f"Entities selected: {bullets}\n")
    print(final[:500])  # First 500 chars of answer


Query: What genes are associated with breast cancer?

Tool: FindGenesForDisease | Input: breast cancer
Entities selected: 20

## BRIEF SUMMARY

Breast cancer is associated with a diverse set of genes spanning receptor pathways, tumor suppressors, and cell cycle regulators (combined evidence across 4,000+ documents). The strongest associations are with HER2 family members (1,068+ documents), estrogen receptor (479 documents), and BRCA genes (154+ documents), reflecting the major molecular subtypes and therapeutic targets in breast cancer.

## SELECTION RATIONALE

I'm selecting genes with >60 documents to capture the mos

Query: List chemicals related to breast cancer

Tool: FindEntityNeighbors | Input: breast cancer, Type: chemical
Entities selected: 15

## BRIEF SUMMARY

Breast cancer is associated with a diverse range of chemicals spanning hormonal agents, targeted therapeutics, and chemotherapy drugs (combined evidence across 20+ results). The strongest associations include estrogen

## Hallucination Check

Cross-check the final answer against tool output to verify the agent only uses KG data.

**Verified automatically:**
- Entity names appear in tool output (with hyphen normalization — NER surface forms like "pdl1" may differ from standard notation "PD-L1" due to inconsistent hyphenation in source text)

**Requires manual inspection:**
- Document counts (LLM may round or summarize)
- Evidence text (LLM may paraphrase)

In [5]:
def _normalize_entity(name: str) -> str:
    """Normalize for comparison: lowercase, strip hyphens (PD-L1 vs pdl1)."""
    return re.sub(r'-', '', name.lower().strip())

def check_for_hallucination(response: dict) -> dict:
    """Verify every entity in final answer appears in tool output."""
    import re
    
    messages = response["messages"]
    final_content = messages[-1].content if messages else ""
    
    # Extract ground truth: entity names from tool outputs (normalized)
    ground_truth = set()
    for msg in messages:
        if msg.__class__.__name__ == "ToolMessage":
            pattern = re.compile(r'ENTITY NAME \(use this exact name\): (.+)$', re.MULTILINE)
            for match in pattern.finditer(msg.content):
                ground_truth.add(_normalize_entity(match.group(1)))
    
    # Check each bullet point in final answer
    verified, hallucinated = [], []
    for line in final_content.split('\n'):
        stripped = line.strip()
        if not stripped.startswith('-'):
            continue
        
        # Extract entity name before separator (now --)
        if '--' in line:
            # Remove leading dash, bold markers, extract name before --
            entity = line.split('--')[0].replace('-', '', 1).replace('**', '').strip()
            entity = entity.split('(')[0].strip()  # Remove parentheticals
            
            if _normalize_entity(entity) in ground_truth:
                verified.append(entity)
            else:
                hallucinated.append(entity)
    
    # Print summary
    print("HALLUCINATION CHECK")
    print("-" * 40)
    print(f"Verified:     {len(verified)}")
    print(f"Hallucinated: {len(hallucinated)}")
    if hallucinated:
        print(f"\nNot in KG: {hallucinated}")
    
    return {"verified": len(verified), "hallucinated": len(hallucinated)}

check_for_hallucination(response)

HALLUCINATION CHECK
----------------------------------------
Verified:     8
Hallucinated: 0


{'verified': 8, 'hallucinated': 0}

---

## Batch Analysis

Run multiple queries to measure aggregate behavior:
- **Tool selection patterns**: Frequency of tools used per query type
- **Selection rate**: Fraction of KG results included in final answers
- **Hallucination rate**: Frequency of entities cited that are not in KG

In [6]:
# Batch analysis framework
import re
from dataclasses import dataclass, field
from collections import defaultdict

@dataclass
class QueryResult:
    """Metrics from a single agent query."""
    query: str
    tools_called: list[str] = field(default_factory=list)
    kg_results_count: int = 0
    selected_entities_count: int = 0
    hallucinated_entities: list[str] = field(default_factory=list)

def analyze_response(query: str, response: dict) -> QueryResult:
    """Extract metrics from an agent response."""
    messages = response.get("messages", [])
    
    # Extract tools
    tools_called = []
    for msg in messages:
        if msg.__class__.__name__ == "AIMessage" and hasattr(msg, 'tool_calls'):
            tools_called.extend(tc.get('name', 'unknown') for tc in (msg.tool_calls or []))
    
    # Extract ground truth: unique entity names from tool outputs (normalized)
    ground_truth = set()
    for msg in messages:
        if msg.__class__.__name__ == "ToolMessage":
            for match in re.finditer(r'ENTITY NAME \(use this exact name\): (.+)$', msg.content, re.MULTILINE):
                ground_truth.add(_normalize_entity(match.group(1)))
    
    # Check final answer: count selections and check hallucinations
    final_content = messages[-1].content if messages else ""
    
    # Extract only DETAILED RESULTS section
    detailed_section = final_content.split("## DETAILED RESULTS")[-1] if "## DETAILED RESULTS" in final_content else ""
    
    hallucinated = []
    selected_count = 0
    
    # Parse each line in detailed section only
    for line in detailed_section.split('\n'):
        if line.strip().startswith('-') and '--' in line:
            selected_count += 1
            entity = line.split('--')[0].replace('-', '', 1).replace('**', '').strip()
            entity = entity.split('(')[0].strip()
            if _normalize_entity(entity) not in ground_truth:
                hallucinated.append(entity)
    
    return QueryResult(
        query=query,
        tools_called=tools_called,
        kg_results_count=len(ground_truth),
        selected_entities_count=selected_count,
        hallucinated_entities=hallucinated
    )

# Test queries covering different tool types
test_queries = [
    "What genes are associated with breast cancer?",
    "What drugs target HER2?",
    "What diseases are associated with BRCA1?",
    "What biological processes are involved in breast cancer?",
    "What is the relationship between estrogen receptor and breast cancer?",
    "What genes are linked to triple-negative breast cancer?",
    "Explain the relationship between HER2 and breast cancer",
]

# Run batch
print(f"Running {len(test_queries)} queries...")
print("-" * 60)

results: list[QueryResult] = []
for i, q in enumerate(test_queries, 1):
    print(f"[{i}/{len(test_queries)}] {q[:50]}...", end=" ")
    try:
        resp = agent.agent.invoke({"messages": [HumanMessage(content=q)]})
        r = analyze_response(q, resp)
        results.append(r)
        status = "OK" if not r.hallucinated_entities else "WARN"
        print(f"{status} | {r.tools_called} | {r.selected_entities_count}/{r.kg_results_count}")
    except Exception as e:
        print(f"ERROR: {e}")

print("-" * 60)
print(f"Completed: {len(results)}/{len(test_queries)}")

Running 7 queries...
------------------------------------------------------------
[1/7] What genes are associated with breast cancer?... Tool: FindGenesForDisease | Input: breast cancer
OK | ['FindGenesForDisease'] | 13/20
[2/7] What drugs target HER2?... Tool: FindEntityNeighbors | Input: HER2, Type: chemical
OK | ['FindEntityNeighbors'] | 6/20
[3/7] What diseases are associated with BRCA1?... Tool: FindDiseasesForGene | Input: BRCA1
OK | ['FindDiseasesForGene'] | 7/10
[4/7] What biological processes are involved in breast c... Tool: FindEntityNeighbors | Input: breast cancer, Type: biological_process
OK | ['FindEntityNeighbors'] | 5/5
[5/7] What is the relationship between estrogen receptor... Tool: ExplainRelationship | Input: estrogen receptor, breast cancer
OK | ['ExplainRelationship'] | 1/1
[6/7] What genes are linked to triple-negative breast ca... Tool: FindGenesForDisease | Input: triple-negative breast cancer
OK | ['FindGenesForDisease'] | 17/20
[7/7] Explain the relationship

### Summary Statistics

In [7]:
# Summary statistics
import pandas as pd

def compute_summary_stats(results: list[QueryResult]) -> pd.DataFrame:
    """Compute aggregate statistics from query results."""
    total_selected = sum(r.selected_entities_count for r in results)
    total_hallucinated = sum(len(r.hallucinated_entities) for r in results)
    total_unique_entities = sum(r.kg_results_count for r in results)
    queries_with_issues = sum(1 for r in results if r.hallucinated_entities)
    total_tool_calls = sum(len(r.tools_called) for r in results)
    
    # Calculate rates safely
    selection_rate = f"{total_selected/total_unique_entities:.1%}" if total_unique_entities else "N/A"
    error_rate = f"{total_hallucinated/total_selected:.1%}" if total_selected else "0.0%"
    avg_tools_per_query = f"{total_tool_calls/len(results):.1f}" if results else "N/A"
    
    return pd.DataFrame({
        "Metric": [
            "Total Queries",
            "Total Tool Calls",
            "Avg Tools/Query",
            "",
            "Unique KG Entities",
            "Entities Selected",
            "Selection Rate",
            "",
            "Hallucinated Entities",
            "Queries with Issues",
            "Entity Error Rate"
        ],
        "Value": [
            len(results),
            total_tool_calls,
            avg_tools_per_query,
            "",
            total_unique_entities,
            total_selected,
            selection_rate,
            "",
            total_hallucinated,
            f"{queries_with_issues}/{len(results)}",
            error_rate
        ]
    })

if results:
    print("AGGREGATE STATISTICS")
    print("=" * 50)
    print(compute_summary_stats(results).to_string(index=False))
    
    # Tool distribution
    print("\n" + "=" * 50)
    print("TOOL USAGE DISTRIBUTION")
    print("-" * 50)
    tool_counts = defaultdict(int)
    for r in results:
        for tool in r.tools_called:
            tool_counts[tool] += 1
    
    total_calls = sum(tool_counts.values())
    for tool, count in sorted(tool_counts.items(), key=lambda x: -x[1]):
        pct = count/total_calls*100
        print(f"  {tool:30} {count:3} ({pct:5.1f}%)")
    
    # Issues section
    print("\n" + "=" * 50)
    print("HALLUCINATION CHECK")
    print("-" * 50)
    issues = [r for r in results if r.hallucinated_entities]
    if issues:
        print(f"Found {len(issues)} queries with hallucinations:\n")
        for r in issues:
            print(f"  Query: {r.query[:60]}...")
            print(f"  Hallucinated: {', '.join(r.hallucinated_entities)}\n")
    else:
        print("[OK] No hallucinations detected - all entities grounded in KG")
else:
    print("Run batch analysis first.")

AGGREGATE STATISTICS
               Metric Value
        Total Queries     7
     Total Tool Calls     7
      Avg Tools/Query   1.0
                           
   Unique KG Entities    77
    Entities Selected    50
       Selection Rate 64.9%
                           
Hallucinated Entities     0
  Queries with Issues   0/7
    Entity Error Rate  0.0%

TOOL USAGE DISTRIBUTION
--------------------------------------------------
  FindGenesForDisease              2 ( 28.6%)
  FindEntityNeighbors              2 ( 28.6%)
  ExplainRelationship              2 ( 28.6%)
  FindDiseasesForGene              1 ( 14.3%)

HALLUCINATION CHECK
--------------------------------------------------
[OK] No hallucinations detected - all entities grounded in KG


---

## Conclusions

### Testing Methodology

Evaluated on 7 breast cancer-focused queries against a KG built from 20K PubMed abstracts (query: "breast cancer AND targeted therapy").

### Key Findings

1. **Tool Selection is Reliable**: The agent routes queries to appropriate tools based on question structure. Gene-disease queries use `FindGenesForDisease`/`FindDiseasesForGene`, chemical queries use `FindEntityNeighbors`, and relationship queries use `ExplainRelationship`.

2. **Intelligent Filtering**: The agent selected a subset of returned entities, showing selectivity rather than including all KG results.

3. **Zero Entity Hallucinations**: All cited entities were verified to exist in tool output (0% error rate). Entity name comparison uses hyphen-aware normalization to handle inconsistent NER surface forms (e.g., KG stores "pdl1" from source text while the LLM correctly writes "PD-L1").

### Implications

- **Strong entity grounding**: All entity names trace back to explicit "ENTITY NAME" fields in tool output
- **Transparent reasoning**: Tool calls are logged and displayed, making entity selection decisions inspectable

### Limitations

- **Entity name normalization**: NER surface forms may differ from standard biomedical nomenclature (hyphens, suffixes like "protein"). The hallucination checker normalizes hyphens but cannot catch all surface-form mismatches.
- **Document accuracy**: Document IDs and document counts were not verified against source data.
- **Evidence quality**: Description accuracy and appropriate paraphrasing require manual review.
- **Non-determinism**: LLM outputs vary between runs; results above are from a single evaluation pass.

In [8]:
# Cleanup
driver.close()
print("Neo4j connection closed")

Neo4j connection closed


---

## Appendix

Sankey diagram showing: Query Type $\rightarrow$ Tool Selected $\rightarrow$ Grounding Outcome

![Sankey Flow](sankey.png)

In [10]:
# Sankey diagram: Query Type -> Tool -> Outcome
import plotly.graph_objects as go

# Color scheme
COLOR_QUERY = "#636EFA"   # Blue
COLOR_TOOL = "#EF553B"    # Red
COLOR_GROUNDED = "#00CC96"  # Green
COLOR_REVIEW = "#FFA15A"   # Orange
COLOR_NO_SELECTION = "#AB63FA"  # Purple

def classify_query_type(query: str) -> str:
    """Simple classification for visualization purposes."""
    q = query.lower()
    
    if "explain" in q or "relationship" in q:
        return "Relationship"
    elif "gene" in q:
        return "Gene Query"
    elif "drug" in q or "chemical" in q:
        return "Chemical Query"
    elif "disease" in q:
        return "Disease Query"
    else:
        return "General"

def create_sankey_diagram(results: list[QueryResult]) -> go.Figure:
    """Create Sankey diagram showing query->tool->outcome flow."""
    flows = defaultdict(int)
    
    for r in results:
        query_type = classify_query_type(r.query)
        for tool in set(r.tools_called):
            flows[(query_type, tool)] += 1
            outcome = "Needs Review" if r.hallucinated_entities else \
                      "Grounded" if r.selected_entities_count > 0 else "No Selection"
            flows[(tool, outcome)] += 1

    # Build nodes
    query_types = sorted(set(classify_query_type(r.query) for r in results))
    tools = sorted(set(t for r in results for t in r.tools_called))
    outcomes = ["Grounded", "Needs Review", "No Selection"]
    
    all_nodes = query_types + tools + [o for o in outcomes if any(o in str(k) for k in flows.keys())]

    # Node colors using constants
    node_colors = (
        [COLOR_QUERY] * len(query_types) +
        [COLOR_TOOL] * len(tools) +
        [COLOR_GROUNDED if "Grounded" in o else COLOR_REVIEW if "Review" in o else COLOR_NO_SELECTION
         for o in outcomes if any(o in str(k) for k in flows.keys())]
    )

    # Build links
    sources, targets, values, link_colors = [], [], [], []
    for (src, tgt), count in flows.items():
        if src in all_nodes and tgt in all_nodes:
            sources.append(all_nodes.index(src))
            targets.append(all_nodes.index(tgt))
            values.append(count)
            # Link colors with transparency
            if "Grounded" in str(tgt):
                link_colors.append("rgba(0, 204, 150, 0.4)")
            elif "Review" in str(tgt):
                link_colors.append("rgba(255, 161, 90, 0.4)")
            else:
                link_colors.append("rgba(200, 200, 200, 0.4)")

    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=20, thickness=20, line=dict(color="black", width=0.5),
                  label=all_nodes, color=node_colors),
        link=dict(source=sources, target=targets, value=values, color=link_colors)
    )])
    fig.update_layout(title_text="Agent Reasoning Flow", font_size=12, height=500, width=1000)
    return fig

# create_sankey_diagram(results).show()